# Practice Notebook — Input-Output Analysis of a Regional Economy

**Based on:** *Python for Engineering and Scientific Computing*, Chapter 3 (NumPy) — the
same `array()`, `@`, `np.linalg.solve()`, and statistical-function toolkit used for the
electrical-network and lightning-protection examples, applied here to a **Leontief
input-output economic model**.

## Learning objectives
By completing this notebook you will practice:
1. Building a coefficient matrix and vector from a word problem (Section 3.1.2)
2. Solving a linear system `A·x = b` with `np.linalg.solve()` (Section 3.4.1)
3. Using matrix multiplication `@` to derive secondary results (Section 3.3)
4. Simulating uncertainty with `np.random.normal()` (Section 3.1.5)
5. Summarizing simulated results with `np.mean`, `np.std`, `np.amin`, `np.amax`, `np.where`
6. Reasoning about **when this kind of model is, and is not, appropriate to use**

## How to use this notebook
- Every task cell contains a `# TODO` and a docstring-style hint. Replace `None` /
  `...` with working code.
- Run cells top to bottom — later tasks depend on variables created earlier.
- Each task has an `assert` sanity check directly below it. If the assert fails,
  your code isn't finished yet — that's expected, it's not a bug in the notebook.
- Don't peek at the cheat sheet until you've tried the task yourself. The cheat
  sheet mirrors this notebook task-by-task, so you can check your work after.
- Section 7 ("Limitations") has no code — it is short-answer reflection. Do not
  skip it: it is the most important section for using this model responsibly.


## Setup
Run this cell first. It imports the libraries used throughout this notebook.

In [ ]:
import numpy as np
from numpy.linalg import solve

np.set_printoptions(precision=2, suppress=True)


---
## Task 1 — Define the economy

A regional economy has three sectors: **Agriculture**, **Manufacturing**, and
**Services**. The *technical coefficient matrix* `A` tells you how many dollars of
input from sector *i* are needed to produce one dollar of output in sector *j*
(row = supplying sector, column = using sector):

| supplies → uses | Agriculture | Manufacturing | Services |
|---|---|---|---|
| Agriculture     | 0.10 | 0.20 | 0.05 |
| Manufacturing   | 0.30 | 0.10 | 0.15 |
| Services        | 0.15 | 0.25 | 0.10 |

**TODO:** Build `sectors` (a list of the 3 names) and `A` (a 3×3 NumPy array) from
the table above. See Listing 3.3 / 3.16 for how `np.array()` builds a matrix from
nested lists.

In [ ]:
sectors = None  # TODO: list of 3 sector name strings, in table order

A = None  # TODO: np.array(...) built from the table above (rows = supplying sector)

# --- sanity check, do not edit ---
assert isinstance(sectors, list) and len(sectors) == 3
assert isinstance(A, np.ndarray) and A.shape == (3, 3)
print("Sectors:", sectors)
print("Technical coefficient matrix A:\n", A)


---
## Task 2 — Define final demand

Final demand (consumers, exports, government) for the year is:
Agriculture = **100**, Manufacturing = **150**, Services = **120** (millions of $).

**TODO:** Build the vector `d`.

In [ ]:
d = None  # TODO: np.array([...]) with the three final-demand values, in the same order as `sectors`

# --- sanity check ---
assert isinstance(d, np.ndarray) and d.shape == (3,)
print("Final demand vector d:", d)


---
## Task 3 — Solve the Leontief system

The Leontief model says total output `x` must satisfy:

`x = A·x + d`  →  `(I − A)·x = d`

This has exactly the same shape as Listing 3.16 (`solve(A, b)`) and Listing 3.18
(`linalg.solve(G, I)`).

**TODO:**
1. Build the 3×3 identity matrix `I3` (see `np.eye()`, used implicitly via matrix
   algebra in the chapter — check the NumPy docs if you haven't used `np.eye`
   before).
2. Build `system_matrix = I3 - A`.
3. Solve for `x` using `solve(system_matrix, d)`.

In [ ]:
I3 = None            # TODO: 3x3 identity matrix
system_matrix = None # TODO: I3 - A
x = None              # TODO: solve(system_matrix, d)

# --- sanity check ---
assert I3.shape == (3, 3) and np.allclose(np.diag(I3), 1)
assert x.shape == (3,)
for s, val in zip(sectors, x):
    print(f"{s:14s}: {val:8.2f}")


---
## Task 4 — Inter-industry transaction matrix

The dollar flow sector *i* sells to sector *j* is `Z[i, j] = A[i, j] * x[j]`. In
matrix form this is `Z = A @ diag(x)` — matrix multiplication exactly like
Listing 3.12/3.13 (`A@B`) and `np.diag()` to turn the output vector into a diagonal
matrix.

**TODO:** Compute `Z`.

In [ ]:
Z = None  # TODO: A @ np.diag(x)

# --- sanity check ---
assert Z.shape == (3, 3)
print("Inter-industry transaction matrix Z:\n", np.round(Z, 2))


---
## Task 5 — Simulate demand uncertainty (Monte Carlo)

Final demand is never known with certainty. Simulate **1,000 scenarios** where each
sector's demand is normally distributed around its base value from Task 2, with
these standard deviations: Agriculture ±15, Manufacturing ±25, Services ±20.

This mirrors Listing 3.6 (`np.random.normal(mean, std, size=...)`).

**TODO:**
1. Set `np.random.seed(1)` so your results are reproducible.
2. Build `d_sim`, a `(1000, 3)` array — one row per scenario, one column per
   sector — using `np.random.normal()` once per sector and `np.column_stack()`
   to combine them.

In [ ]:
np.random.seed(1)
n_scenarios = 1000

d_sim = None  # TODO: shape (1000, 3) array of simulated demand

# --- sanity check ---
assert d_sim.shape == (n_scenarios, 3)
print(d_sim[:5])  # first 5 simulated scenarios


---
## Task 6 — Solve every scenario at once

For each simulated demand row `d_sim[k]`, the corresponding output is
`x_sim[k] = (I - A)^-1 @ d_sim[k]`.

Rather than looping 1,000 times, use `np.linalg.inv()` once to get the inverse of
`system_matrix`, then apply it to every row with a single matrix multiplication:
`x_sim = d_sim @ inv_system.T`.

**TODO:** Compute `inv_system` and `x_sim`.

In [ ]:
inv_system = None  # TODO: np.linalg.inv(system_matrix)
x_sim = None        # TODO: d_sim @ inv_system.T

# --- sanity check ---
assert x_sim.shape == (n_scenarios, 3)
print("First 3 simulated outputs:\n", x_sim[:3])


---
## Task 7 — Summarize the simulation

For each sector (each column of `x_sim`), compute the mean, standard deviation,
minimum, and maximum output using the statistical functions from Section 3.1.5
(`np.mean`, `np.std`, `np.amin`, `np.amax`).

Then find:
- the **scenario index** with the highest *total* output across all sectors
  (`np.sum(..., axis=1)` + `np.where`)
- the **most volatile sector**, using the coefficient of variation
  `std / mean` for each column, and `np.argmax()` to find the largest one

**TODO:** Fill in the blanks below.

In [ ]:
for i, s in enumerate(sectors):
    col = x_sim[:, i]
    mean = None  # TODO
    std = None   # TODO
    mn = None    # TODO
    mx = None    # TODO
    print(f"{s:14s} mean={mean:8.2f}  std={std:6.2f}  min={mn:8.2f}  max={mx:8.2f}")

total_output = None  # TODO: np.sum(x_sim, axis=1)
max_index = None      # TODO: np.where(total_output == np.amax(total_output))
print("\nHighest-output scenario index:", max_index[0][0])

cv = None  # TODO: np.std(x_sim, axis=0) / np.mean(x_sim, axis=0)
most_volatile = sectors[np.argmax(cv)]
print("Coefficients of variation:", np.round(cv, 3))
print("Most volatile sector:", most_volatile)


---
## Section 8 — Limitations (reflection, no code)

This model is a **linear, static, fixed-technology** approximation of an economy.
Before presenting results from a notebook like this to anyone, answer the
questions below in your own words (2–3 sentences each).

**When is this model a reasonable choice?**
- Short-run policy questions ("if government spending on Services rises by $20M,
  how does that ripple through Agriculture and Manufacturing?")
- Economies/industries where the mix of inputs per unit of output is genuinely
  close to fixed over the period you're studying
- Getting a first-order, directionally correct estimate quickly, to be refined
  with a richer model afterward

**When is it NOT recommended?**
- Long-run forecasting, where technology and input ratios change (the whole
  model rests on `A` being constant)
- Situations with strong price effects, substitution between inputs, or
  economies of scale — the model is linear and has no prices or behavior in it
- Economies with structural breaks (financial crises, wars, pandemics, sudden
  supply shocks) — the coefficients estimated from "normal" data no longer apply
- Treating the Monte Carlo output as a real probability forecast. The normal
  distributions in Task 5 were **assumed**, not estimated from real data — the
  spread only tells you about the assumptions you fed in, not the real economy.

1. In your own words, what does the assumption "`A` is constant" mean economically,
   and give one real-world event that would break it.
2. Why can `x = (I-A)^-1 d` return a *negative* number for some sector under some
   random demand draw, and what would that mean physically? Is a negative output
   ever a sensible real-world answer?
3. Name one economic question this notebook's model is well suited to answer, and
   one it is not.
